In [ ]:
import boto3
import pandas as pd
import io
from datetime import datetime, timedelta
from sqlalchemy import create_engine, text


def get_date_range(start_date=None, end_date=None):

    if start_date is None and end_date is None:
        return [datetime.utcnow().strftime("%Y%m%d")]

    if start_date and end_date is None:
        return [datetime.strptime(start_date, "%Y-%m-%d").strftime("%Y%m%d")]

    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")

    dates = []
    d = start

    while d <= end:
        dates.append(d.strftime("%Y%m%d"))
        d += timedelta(days=1)

    return dates


def main(
    aws_access_key,
    aws_secret_key,
    bucket_name,
    base_folder,

    db_host,
    db_name,
    db_user,
    db_password,
    table_name,

    start_date=None,
    end_date=None,

    region="ap-south-1"
):

    print("\nStarting S3 → PostgreSQL ingestion pipeline")

    dates = get_date_range(start_date, end_date)

    print("Processing dates:", dates)

    s3 = boto3.client(
        "s3",
        aws_access_key_id=aws_access_key,
        aws_secret_access_key=aws_secret_key,
        region_name=region
    )

    engine = create_engine(
        f"postgresql://{db_user}:{db_password}@{db_host}:5432/{db_name}"
    )

    conn = engine.connect()

    # ───────── Pipeline Start Audit ─────────
    start_time = datetime.utcnow()

    run_insert = conn.execute(text("""
        INSERT INTO bronze.pipeline_run_audit
        (pipeline_name, source_system, run_date, start_time, status)
        VALUES ('billing_s3_loader','aws_s3',CURRENT_DATE,:start_time,'RUNNING')
        RETURNING run_id
    """), {"start_time": start_time})

    run_id = run_insert.fetchone()[0]

    print(f"Pipeline run_id = {run_id}")

    total_records = 0
    files_processed = 0
    all_dfs = []

    try:

        for d in dates:

            folder = f"{base_folder}/billing_{d}"

            print(f"\nScanning folder: {folder}")

            response = s3.list_objects_v2(
                Bucket=bucket_name,
                Prefix=folder
            )

            files = [
                obj["Key"]
                for obj in response.get("Contents", [])
                if obj["Key"].endswith(".parquet")
            ]

            if not files:
                print("No parquet files found")
                continue

            print(f"Found {len(files)} parquet files")

            for file_key in files:

                file_start = datetime.utcnow()

                print("Reading:", file_key)

                try:

                    obj = s3.get_object(
                        Bucket=bucket_name,
                        Key=file_key
                    )

                    df = pd.read_parquet(
                        io.BytesIO(obj["Body"].read())
                    )

                    records = len(df)

                    all_dfs.append(df)

                    total_records += records
                    files_processed += 1

                    file_end = datetime.utcnow()

                    conn.execute(text("""
                        INSERT INTO bronze.file_load_audit
                        (run_id,file_name,s3_path,file_date,records_loaded,
                         load_status,load_start_time,load_end_time)
                        VALUES
                        (:run_id,:file_name,:s3_path,:file_date,:records,
                         'SUCCESS',:start,:end)
                    """), {
                        "run_id": run_id,
                        "file_name": file_key.split("/")[-1],
                        "s3_path": file_key,
                        "file_date": datetime.strptime(d,"%Y%m%d").date(),
                        "records": records,
                        "start": file_start,
                        "end": file_end
                    })

                except Exception as e:

                    file_end = datetime.utcnow()

                    conn.execute(text("""
                        INSERT INTO bronze.file_load_audit
                        (run_id,file_name,s3_path,file_date,records_loaded,
                         load_status,error_message,load_start_time,load_end_time)
                        VALUES
                        (:run_id,:file_name,:s3_path,:file_date,0,
                         'FAILED',:error,:start,:end)
                    """), {
                        "run_id": run_id,
                        "file_name": file_key.split("/")[-1],
                        "s3_path": file_key,
                        "file_date": datetime.strptime(d,"%Y%m%d").date(),
                        "error": str(e),
                        "start": file_start,
                        "end": file_end
                    })

                    print("File failed:", file_key)

        if not all_dfs:
            print("\nNo data found. Exiting.")

        else:

            final_df = pd.concat(all_dfs, ignore_index=True)

            print(f"\nTotal records collected: {len(final_df)}")

            print("\nLoading into PostgreSQL...")

            final_df.to_sql(
                table_name,
                engine,
                if_exists="append",
                index=False
            )

        end_time = datetime.utcnow()

        conn.execute(text("""
            UPDATE bronze.pipeline_run_audit
            SET status='SUCCESS',
                end_time=:end_time,
                files_processed=:files,
                total_records=:records
            WHERE run_id=:run_id
        """),{
            "end_time": end_time,
            "files": files_processed,
            "records": total_records,
            "run_id": run_id
        })

        print("\n✓ Data successfully loaded to PostgreSQL")

    except Exception as e:

        end_time = datetime.utcnow()

        conn.execute(text("""
            UPDATE bronze.pipeline_run_audit
            SET status='FAILED',
                end_time=:end_time,
                error_message=:error
            WHERE run_id=:run_id
        """),{
            "end_time": end_time,
            "error": str(e),
            "run_id": run_id
        })

        print("Pipeline failed:", e)

    finally:
        conn.close()

: 

In [ ]:

# Example execution
if __name__ == "__main__":

    main(

        aws_access_key="YOUR_ACCESS_KEY",
        aws_secret_key="YOUR_SECRET_KEY",

        bucket_name="retail-data-lake",
        base_folder="retail-data",

        db_host="localhost",
        db_name="retail_db",
        db_user="postgres",
        db_password="password",

        table_name="silver_billing_transactions",

        start_date=None,
        end_date=None
    )